# DPO Training Example Notebook

This notebook demonstrates how to use the GPT2Wrapper, DPO utilities, and Trainer for both SFT and DPO training.

## Setup

Make sure to install dependencies first:
```bash
pip install -r requirements.txt
```

## 1. Import Required Modules

In [ ]:
import sys
sys.path.insert(0, 'src')

import torch
import numpy as np
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# Import our custom modules
from model import GPT2Wrapper
from dpo_utils import compute_dpo_loss, get_token_log_probs
from trainer import Trainer

print(' All imports successful!')
print(f' CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f' GPU: {torch.cuda.get_device_name(0)}')

## 2. Initialize Models

Create the policy model and a frozen reference model for DPO.

In [ ]:
# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# Initialize policy model
print('\nLoading GPT-2 policy model...')
policy_model = GPT2Wrapper('gpt2', device=device)
print(f' Policy model loaded: {policy_model.get_model_name()}')
print(f' Device: {policy_model.get_device()}')

# Get reference model
print('\nCreating reference model...')
ref_model = policy_model.get_reference_model()
print(f' Reference model created (frozen: {not next(ref_model.model.parameters()).requires_grad})')

## 3. Test Model Forward Pass

Verify that the models return correct logit shapes.

In [ ]:
# Create synthetic batch
batch_size = 2
seq_length = 10
input_ids = torch.randint(0, 50257, (batch_size, seq_length))
attention_mask = torch.ones_like(input_ids)

print('Test forward pass...')
with torch.no_grad():
    policy_logits = policy_model(input_ids, attention_mask)
    ref_logits = ref_model(input_ids, attention_mask)

print(f' Policy logits shape: {policy_logits.shape}')
print(f' Reference logits shape: {ref_logits.shape}')
print(f' Expected shape: ({batch_size}, {seq_length}, 50257)')

assert policy_logits.shape == (batch_size, seq_length, 50257), 'Policy logits shape mismatch!'
assert ref_logits.shape == (batch_size, seq_length, 50257), 'Reference logits shape mismatch!'
print('\n All shape checks passed!')

## 4. Test DPO Loss Computation

Verify that the DPO loss function works correctly.

In [ ]:
# Create synthetic labels
winner_labels = torch.randint(0, 50257, (batch_size, seq_length))
loser_labels = torch.randint(0, 50257, (batch_size, seq_length))

# Add some padding
mask = torch.rand(batch_size, seq_length) > 0.8
winner_labels[mask] = -100
loser_labels[mask] = -100

print('Computing DPO loss...')
with torch.no_grad():
    policy_logits = policy_model(input_ids, attention_mask)
    ref_logits = ref_model(input_ids, attention_mask)
    
    loss, metrics = compute_dpo_loss(
        policy_logits=policy_logits,
        ref_logits=ref_logits,
        winner_labels=winner_labels,
        loser_labels=loser_labels,
        beta=0.1
    )

print(f' DPO Loss: {loss.item():.6f}')
print(f'\nMetrics:')
for key, value in metrics.items():
    print(f'  - {key}: {value:.6f}')

## 5. Test Token Log Probabilities Extraction

Verify the get_token_log_probs utility function.

In [ ]:
print('Testing get_token_log_probs...')

logits = torch.randn(batch_size, seq_length, 50257)
labels = torch.randint(0, 50257, (batch_size, seq_length))

log_probs = get_token_log_probs(logits, labels)

print(f' Input logits shape: {logits.shape}')
print(f' Input labels shape: {labels.shape}')
print(f' Output log_probs shape: {log_probs.shape}')
print(f' Log prob range: [{log_probs.min():.4f}, {log_probs.max():.4f}]')

assert log_probs.shape == (batch_size, seq_length), 'Log probs shape mismatch!'
print('\n Log probability extraction passed!')

## 6. Create Synthetic Dataset

Create a simple dataset for demonstration purposes.

In [ ]:
class SyntheticDataset(Dataset):
    """Simple synthetic dataset for demonstration."""
    def __init__(self, num_samples=10, seq_length=10, mode='sft'):
        self.num_samples = num_samples
        self.seq_length = seq_length
        self.mode = mode
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        if self.mode == 'sft':
            return {
                'input_ids': torch.randint(0, 50257, (self.seq_length,)),
                'attention_mask': torch.ones(self.seq_length),
                'labels': torch.randint(0, 50257, (self.seq_length,))
            }
        else:  # dpo
            return {
                'winner_input_ids': torch.randint(0, 50257, (self.seq_length,)),
                'winner_attention_mask': torch.ones(self.seq_length),
                'winner_labels': torch.randint(0, 50257, (self.seq_length,)),
                'loser_input_ids': torch.randint(0, 50257, (self.seq_length,)),
                'loser_attention_mask': torch.ones(self.seq_length),
                'loser_labels': torch.randint(0, 50257, (self.seq_length,))
            }

print('Creating datasets...')
sft_dataset = SyntheticDataset(num_samples=8, seq_length=10, mode='sft')
dpo_dataset = SyntheticDataset(num_samples=8, seq_length=10, mode='dpo')

sft_loader = DataLoader(sft_dataset, batch_size=2, shuffle=True)
dpo_loader = DataLoader(dpo_dataset, batch_size=2, shuffle=True)

print(f' SFT dataset: {len(sft_dataset)} samples')
print(f' DPO dataset: {len(dpo_dataset)} samples')
print(f' SFT DataLoader: {len(sft_loader)} batches')
print(f' DPO DataLoader: {len(dpo_loader)} batches')

## 7. Initialize Trainer

Set up the unified trainer for both SFT and DPO modes.

In [ ]:
print('Initializing Trainer...')
trainer = Trainer(
    policy_model=policy_model,
    ref_model=ref_model,
    learning_rate=5e-5,
    device=device,
    beta=0.1
)
print(' Trainer initialized!')
print(f'  - Learning rate: 5e-5')
print(f'  - Beta: 0.1')
print(f'  - Device: {device}')

## 8. Test SFT Training (Single Step)

Run a single SFT training step to verify the training loop.

In [ ]:
print('Testing SFT training step...')

# Get a batch
batch = next(iter(sft_loader))

# Move to device
for key in batch:
    batch[key] = batch[key].to(device)

# Run training step
loss, metrics = trainer.sft_train_step(batch)

print(f' SFT training step completed!')
print(f'  - Loss: {loss.item():.6f}')
print(f'  - Metrics:')
for key, value in metrics.items():
    print(f'    * {key}: {value:.6f}')

## 9. Test DPO Training (Single Step)

Run a single DPO training step to verify the DPO training loop.

In [ ]:
print('Testing DPO training step...')

# Get a batch
batch = next(iter(dpo_loader))

# Move to device
for key in batch:
    batch[key] = batch[key].to(device)

# Run training step
loss, metrics = trainer.dpo_train_step(batch)

print(f' DPO training step completed!')
print(f'  - Loss: {loss.item():.6f}')
print(f'  - Metrics:')
for key, value in metrics.items():
    print(f'    * {key}: {value:.6f}')

## 10. Full Training Loop (SFT)

Run a complete SFT training loop for demonstration.

In [ ]:
print('\n' + '='*60)
print('Starting SFT Training (2 epochs)')
print('='*60 + '\n')

sft_results = trainer.train(
    epochs=2,
    dataloader=sft_loader,
    mode='sft'
)

print('\nSFT Training Results:')
print(f'  - Train losses: {sft_results["train_losses"]}')

## 11. Full Training Loop (DPO)

Run a complete DPO training loop for demonstration.

In [ ]:
print('\n' + '='*60)
print('Starting DPO Training (2 epochs)')
print('='*60 + '\n')

# Re-initialize trainer to reset optimizer
trainer = Trainer(
    policy_model=policy_model,
    ref_model=ref_model,
    learning_rate=5e-5,
    device=device,
    beta=0.1
)

dpo_results = trainer.train(
    epochs=2,
    dataloader=dpo_loader,
    mode='dpo'
)

print('\nDPO Training Results:')
print(f'  - Train losses: {dpo_results["train_losses"]}')

## 12. Summary and Testing Report

Print a comprehensive testing report.

In [ ]:
print('\n' + '='*70)
print('COMPREHENSIVE TESTING REPORT')
print('='*70)

print('\n MODULE TESTS PASSED:')
print('  [] model.py - GPT2Wrapper class')
print('      - Forward pass returns correct logits shape')
print('      - Reference model creation works')
print('      - Device management functions')

print('\n  [] dpo_utils.py - DPO loss computation')
print('      - compute_dpo_loss() function works')
print('      - get_token_log_probs() utility function')
print('      - Metric computation')

print('\n  [] trainer.py - Unified training')
print('      - SFT single training step')
print('      - DPO single training step')
print('      - Full SFT training loop (2 epochs)')
print('      - Full DPO training loop (2 epochs)')

print('\n' + '='*70)
print('TRAINING SUMMARY')
print('='*70)
print(f'\nSFT Training (2 epochs):')
print(f'  - Initial loss: {sft_results["train_losses"][0]:.6f}')
print(f'  - Final loss: {sft_results["train_losses"][-1]:.6f}')

print(f'\nDPO Training (2 epochs):')
print(f'  - Initial loss: {dpo_results["train_losses"][0]:.6f}')
print(f'  - Final loss: {dpo_results["train_losses"][-1]:.6f}')

print('\n' + '='*70)
print('ALL TESTS PASSED SUCCESSFULLY!')
print('='*70)